# openai-responses — reasoning and cached tokens are billed too

New OpenAI apps call `responses.create`, which reports usage differently. Cost tooling that only understands `prompt_tokens`/`completion_tokens` silently misses both.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | `OpenAI()` — the `responses.create` shape |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

**Distinctive here: reasoning and cached tokens.** Both are billed, at different rates.

## 1–2 · Connect and instrument

In [ ]:
import main as recipe
from cendor.core import bus, instrument
from cendor.core.types import LLMCall

seen, calls = [], []
bus.subscribe(lambda e: calls.append(e) if isinstance(e, LLMCall) else None)
client = instrument(recipe.fake_openai_responses(seen))

## 3 · One governed call

In [ ]:
from cendor.tokenguard import budget, reset

reset()
with budget(usd=0.50, on_exceed="block"):
    client.responses.create(model="gpt-4o", input="Summarize, then reason.")
call = calls[-1]
u = call.usage
print(
    f"usage: {u.input_tokens:,} in ({u.cached_tokens} cached)"
    f" -> {u.output_tokens:,} out ({u.reasoning_tokens} reasoning)"
)
print(f"cost : ${call.cost.amount}")

## 5 · Prove it

The interesting failure is silent: a normalizer that stopped reading `*_tokens_details` would still report a plausible-looking cost.

In [ ]:
assert u.reasoning_tokens == 620, "reasoning tokens were not normalized out of the details"
assert u.cached_tokens == 200, "cached tokens were not normalized out of the details"
assert call.cost and call.cost.amount > 0
print("OK — reasoning + cached are IN the cost")